# NRW-Wohngebäudebestand — Clustering
**je Gebäudetyp | Bedarf + Dach | Log-Raum-KMeans mit Coverage-Kriterium**

Zentrales Clustering-Notebook. Es liest die gemeinsame Arbeitsdatei
`nrw_buildings.parquet` (aus `01_nrw_pipeline.ipynb`), clustert Strom-, Wärme-
und Dachflächen je Gebäudetyp und schreibt die Cluster-Repräsentanten (geometrische
Mittel) zurück in dieselbe Datei.

Der Clustering-Algorithmus ist der aus dem NL-Pendant
(`netherlands-building-clustering-main`, `05_nl_clustering.ipynb`): eine eigene,
deterministische 1D-Lloyd-Implementierung **im Log-Raum**, deren k nicht per Elbow/Silhouette, sondern über das Coverage-Kriterium
bestimmt wird: k steigt von 1 aufwärts, bis mindestens 80 % der Gebäude
höchstens ±20 % relative Abweichung von ihrem Cluster-Zentrum haben.
Kein Seed, kein Sampling — das Ergebnis ist bei gleicher Eingabe identisch
reproduzierbar.

NRW-spezifische Festlegungen (Datengrundlage/Herleitung):
- der **Parzellen-Filter** (bestätigte Wohngebäude via Wohnbauflächen) als Trainingsmenge,
- der **P99.5-Outlier-Cap** vor dem Bedarfs-Clustering,
- der **Sonderfall SFH/TH-Strom** (nur 2 mögliche Werte, fest verdrahtet — siehe Abschnitt 6),
- die **LOD2-Dach-Klassifikation** (Flachdach < 10° Neigung, Ost+West kombiniert, Nord ausgeschlossen).

| Abschnitt | Aufgabe |
|---|---|
| 1 | Konfiguration |
| 2 | Bestätigte Wohngebäude — Parzellen-Filter |
| 3 | Trainingsmenge laden + Ausreißer-Entfernung |
| 4 | Bedarfsverteilungen |
| 5 | Clustering-Hilfsfunktionen (Log-Raum-KMeans) |
| 6 | Clustering je Gebäudetyp — Bedarf |
| 7 | Gesamtdatensatz-Vergleich |
| 8 | Kombinationszahlen-Vergleich |
| 9 | Abstandsanalyse — Bedarf |
| 10 | Clustergrößen-Validierung |
| 11 | Sanierungszustand-Kreuztabelle |
| 12 | Dachdaten — LOD2 laden und joinen |
| 13 | Dach-Klassifikation |
| 14 | Dachflächen-Verteilungen |
| 15 | Clustering je Gebäudetyp — Dach |
| 16 | Dach-Kombinationen |
| 17 | Dach-Abstandsanalyse |
| 18 | Cluster-Repräsentanten allen Gebäuden zuweisen |
| 19 | In die gemeinsame Arbeitsdatei schreiben |
| 20 | Excel-Zusammenfassung speichern |

## 1. Konfiguration

In [15]:
import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import wkb

warnings.filterwarnings("ignore")

# Relative Pfade — funktioniert auf jedem Rechner, solange alle Dateien im
# Projektstamm liegen. Identisch zum NL-Repositorium: eine gemeinsame
# Arbeitsdatei, die jedes Notebook liest und wieder ueberschreibt.
BASE_DIR          = "."
FILE_GEBAEUDE     = os.path.join(BASE_DIR, "nrw_buildings.parquet")
FILE_WOHN         = os.path.join(BASE_DIR, "katasterdaten", "wohnflaechen_nrw.parquet")
FILE_ROOF         = os.path.join(BASE_DIR, "lod2_nrw_roofs.geoparquet")
OUT_DIR           = os.path.join(BASE_DIR, "nrw_results")

SIZE_CLASSES = ["SFH", "TH", "MFH", "AB"]

FLAT_TILT_THRESHOLD = 10   # Grad — darunter gilt ein Dach als Flachdach

# --- Clustering-Einstellungen (wie 05_nl_clustering.ipynb) ---
# kmeans_mit_abweichungskriterium()
# (Abschnitt 5) startet grundsaetzlich bei k=1 und ist als eigene,
# deterministische Lloyd-Implementierung im Log-Raum ohne Sampling weder auf
# einen Seed noch auf Minibatching angewiesen.
K_MAX = 12                     # obere Grenze fuer die Clusteranzahl
KMEANS_1D_MAX_ITER = 10_000     # Sicherheitsobergrenze fuer Lloyd-Iterationen je Fit
MIN_GEBAEUDE_FUER_CLUSTERING = 50   # darunter bekommt eine Groessenklasse nur 1 Cluster

# --- Coverage-Kriterium ---
COVERAGE_MIN = 0.80    # Mindestanteil Gebaeude, der je (Groessenklasse, Merkmal)
                        # "abgedeckt" sein muss, damit ein k akzeptiert wird
TOLERANZ_BAND = 0.20    # ein Gebaeude gilt als abgedeckt, wenn seine relative
                         # Abweichung vom Cluster-Zentrum hoechstens so gross ist

# Die Zielabdeckung gilt global fuer alle Kombinationen: 80 %. Gezielte
# Ausnahmen sind derzeit nicht noetig -- dieses Dict bleibt als Mechanismus
# fuer die spaetere Feinsteuerung einzelner (Groessenklasse, Merkmal)-
# Kombinationen erhalten (Kandidaten erkennt man im Uebersicht-Sheet der
# Excel-Zusammenfassung an coverage < coverage_ziel).
COVERAGE_MIN_AUSNAHMEN = {}

os.makedirs(OUT_DIR, exist_ok=True)
for sc in SIZE_CLASSES:
    os.makedirs(os.path.join(OUT_DIR, sc), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "gesamt"), exist_ok=True)

print("Konfiguration geladen (eigene 1D-KMeans-Implementierung im Log-Raum, "
      f"K_MAX={K_MAX}, Coverage-Ziel {COVERAGE_MIN:.0%} innerhalb +/-{TOLERANZ_BAND:.0%}).")

Konfiguration geladen (eigene 1D-KMeans-Implementierung im Log-Raum, K_MAX=12, Coverage-Ziel 80% innerhalb +/-20%).


## 2. Bestätigte Wohngebäude — Parzellen-Filter

Nur Gebäude, deren Zentroide in einer Wohnbauflächen-Parzelle liegen, gelten als
bestätigt wohnbaulich. Das gibt dem Bedarfs-Clustering eine saubere Trainingsmenge
— Hotels, Garagen und Industriebauten, die es in den Gebäudedaten bis hierher
geschafft haben, werden hier ausgeschlossen.

Der Filter schränkt nur die **Trainingsmenge** ein. Die Dach-Clusterings und die
Zuweisung der Cluster-Repräsentanten (Abschnitt 18) decken weiterhin alle
Gebäude ab — ein Gebäude außerhalb einer Wohnbauflächen-Parzelle bekommt trotzdem
Cluster-Werte in der Arbeitsdatei.

In [16]:
# Gemeinsame Arbeitsdatei einmal komplett laden — alle Abschnitte arbeiten auf
# dieser einen Kopie und Abschnitt 19 schreibt sie angereichert zurueck.
gebaeude = gpd.read_parquet(FILE_GEBAEUDE)
if gebaeude.geometry.name != "footprint":
    gebaeude = gebaeude.set_geometry("footprint")
gebaeude["centroid"] = gebaeude["footprint"].centroid
print(f'Arbeitsdatei geladen: {len(gebaeude):,} Gebaeude, '
      f'{len(gebaeude.columns)} Spalten')

# Wohnbauflaechen-Parzellen
wohn = pd.read_parquet(FILE_WOHN)
wohn["geometry_decoded"] = wohn["geometry"].apply(
    lambda x: wkb.loads(x) if isinstance(x, bytes) else x
)
wohn_gdf = gpd.GeoDataFrame(wohn, geometry="geometry_decoded", crs="EPSG:25832")
print(f'Parzellen geladen: {len(wohn_gdf):,}  CRS: EPSG:25832')

# Inner Join — nur Gebaeude, deren Zentroid in einer Wohnbauflaeche liegt
matched = gpd.sjoin(
    gebaeude.set_geometry("centroid"), wohn_gdf,
    how="inner", predicate="intersects", rsuffix="lika"
)
confirmed_ids = pd.Index(matched["id"].unique())

print(f'Gebaeude gesamt        : {len(gebaeude):,}')
print(f'Bestaetigt wohnbaulich : {len(confirmed_ids):,}  '
      f'({len(confirmed_ids)/len(gebaeude)*100:.1f}%)')
print()
print("Nach Groessenklasse:")
for sc in SIZE_CLASSES:
    b = (gebaeude["size_class"] == sc).sum()
    a = gebaeude["id"].isin(confirmed_ids).loc[gebaeude["size_class"] == sc].sum()
    behalten = f"  ({a/b*100:.1f}% behalten)" if b > 0 else ""
    print(f"  {sc}: {b:>10,} -> {a:>10,}{behalten}")

Arbeitsdatei geladen: 4,134,282 Gebaeude, 19 Spalten
Parzellen geladen: 4,534,540  CRS: EPSG:25832
Gebaeude gesamt        : 4,134,282
Bestaetigt wohnbaulich : 3,705,346  (89.6%)

Nach Groessenklasse:
  SFH:  2,723,800 ->  2,485,930  (91.3% behalten)
  TH:    507,716 ->    492,547  (97.0% behalten)
  MFH:    897,417 ->    723,908  (80.7% behalten)
  AB:      5,349 ->      4,304  (80.5% behalten)


## 3. Trainingsmenge laden + Ausreißer-Entfernung

`gebaeude` (alle Gebäude) bleibt unangetastet — es wird später für den Dach-Join
und die Zuweisung wiederverwendet. `training` ist die schmalere, bereinigte
Trainingsmenge, ausschließlich für das Fitten der Bedarfs-Cluster.

### Ausreißer-Entfernung wegen unrepräsentativer Cluster

Ohne diese Obergrenze werden die oberen Cluster aufgebläht und unterrepräsentieren
die kleinen, deutlich häufigeren Bedarfe. Cap = P99.5 der Wohnfläche je
Gebäudetyp (das aggressivere P95 wurde getestet und verworfen, weil es zu viele
echte Gebäude entfernt — Details im `schwellenwertvergleich_p95_p99.ipynb`).

In [17]:
# Bestätigt-wohnbauliche Trainingsmenge fuer das Bedarfs-Clustering
training = gebaeude[gebaeude["id"].isin(confirmed_ids)].copy().reset_index(drop=True)
print(f'Nach Wohnbauflaechen-Filter: {len(training):,}  '
      f'({len(training)/len(gebaeude)*100:.1f}% von insgesamt)')
print()
print("Zahl je Gebaeudetyp:")
print(training["size_class"].value_counts().to_string())

# P99.5-Cap je Groessenklasse
caps = {sc: training.loc[training["size_class"] == sc, "Al_m2"].quantile(0.995)
        for sc in SIZE_CLASSES}

# Ausreisser-Entfernung — Gebaeude sind bereits bestaetigt wohnbaulich
n_before = len(training)
training = pd.concat([
    training.loc[(training["size_class"] == sc) & (training["Al_m2"] < caps[sc])]
    for sc in SIZE_CLASSES
], axis=0).reset_index(drop=True)
dropped = n_before - len(training)
if dropped > 0:
    print(f'{dropped:,} Gebaeude durch Ausreisser-Entfernung (P99.5) entfernt')
else:
    print("Keine Ausreisser — alle Gebaeude bleiben")

# Gebaeude ohne Waermewert entfernen (sollte nach Pipeline-Fix 0 sein)
n_before = len(training)
training = training.dropna(subset=["heat_total_kwh"]).copy()
dropped = n_before - len(training)
print(f'Fehlende Waerme: {dropped:,} Gebaeude entfernt' if dropped
      else "Keine fehlenden Waermedaten")

print(f'\nFinale Trainingsmenge: {len(training):,} bestaetigt wohnbauliche Gebaeude')
print(training["size_class"].value_counts().to_string())
print(f'\nStrom gesamt : {training["elec_kwh"].sum()/1e9:.2f} TWh/Jahr')
print(f'BDEW-Referenz: 31.96 TWh/Jahr')

Nach Wohnbauflaechen-Filter: 3,706,689  (89.7% von insgesamt)

Zahl je Gebaeudetyp:
size_class
SFH    2485930
MFH     723908
TH      492547
AB        4304
18,535 Gebaeude durch Ausreisser-Entfernung (P99.5) entfernt
Keine fehlenden Waermedaten

Finale Trainingsmenge: 3,688,154 bestaetigt wohnbauliche Gebaeude
size_class
SFH    2473500
MFH     720288
TH      490084
AB        4282

Strom gesamt : 15.18 TWh/Jahr
BDEW-Referenz: 31.96 TWh/Jahr


## 4. Bedarfsverteilungen

Zeigt die Streuung der Bedarfswerte über die Trainingsmenge.

**Besonderheit der Stromverteilung:** Sie ist komprimiert und diskret. Die
meisten SFH/TH haben 1 Wohnung und 2 Personen →
alle bekommen exakt 2.700 kWh. Das ist ein Artefakt der CO2online-Lookup-Tabelle,
die feste Werte je Wohnungszahl vergibt.

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label, color in zip(
    axes,
    ["elec_kwh", "heat_total_kwh"],
    ["Strom (kWh/Jahr)", "Waerme (kWh/Jahr)"],
    ["steelblue", "tomato"]
):
    sample = training[col].sample(50_000, random_state=42)
    ax.hist(sample, bins=60, color=color, alpha=0.8, edgecolor="white")
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel("Anzahl", fontsize=10)
    ax.set_title(f"Verteilung {label}", fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

plt.suptitle("Bedarfsverteilungen — Stichprobe von 50.000 Gebaeuden", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "bedarfsverteilungen.pdf"), bbox_inches="tight")
plt.close()
print("Hinweis: Die Stromverteilung ist diskret — Effekt der CO2online-Lookup-Tabelle")

Hinweis: Die Stromverteilung ist diskret — Effekt der CO2online-Lookup-Tabelle


## 5. Clustering-Hilfsfunktionen (Log-Raum-KMeans)

1:1 aus `05_nl_clustering.ipynb` (NL-Repositorium) übernommen.

### 5.1 Eigene 1D-KMeans-Implementierung im Log-Raum

**Warum Log-Raum?** Eine absolute Distanz im Log-Raum entspricht einer relativen
Distanz im Originalraum — sie passt damit exakt zur relativen
Abweichungsmetrik des Coverage-Kriteriums. Cluster-Zentren sind geometrische
Mittel (`exp(mean(log(werte)))`), keine arithmetischen.

**Warum kein Elbow/Silhouette?** Der Elbow ist ein geometrisches Kriterium
ohne direkte Aussage darueber, wie gut ein einzelnes Gebaeude durch seinen
Cluster-Repraesentanten dargestellt wird. Das Coverage-Kriterium (k steigt, bis
≥ 80 % der Gebäude innerhalb ±20 % liegen) hat diese Aussage eingebaut: k ist
die kleinste Clusteranzahl, mit der die Simulation "gut genug" wird.

**Warum deterministisch?** Quantil-Initialisierung + Lloyd-Konvergenz ohne
Sampling: kein Seed, kein Zufall — bei gleicher Eingabe identisches Ergebnis,
Cluster-Nummern sind über Runs stabil.

In [19]:
def _kmeans_1d(values, k, n_iter=KMEANS_1D_MAX_ITER):
    """1D-K-Means im Log-Raum: absolute Distanz im Log-Raum entspricht relativer
    Distanz im Originalraum -- passt damit zur relativen Abweichungsmetrik.
    Braucht Werte > 0 (np.log, nicht log1p)."""
    log_values = np.log(values)
    log_centers = np.quantile(log_values, np.linspace(0, 1, k + 2)[1:-1])

    for _ in range(n_iter):
        distances = np.abs(log_values[:, None] - log_centers[None, :])
        labels = distances.argmin(axis=1)
        new_log_centers = np.array([
            log_values[labels == i].mean() if np.any(labels == i) else log_centers[i]
            for i in range(k)
        ])
        if np.allclose(new_log_centers, log_centers):
            break
        log_centers = new_log_centers

    centers = np.exp(log_centers)  # zurueck in den Originalraum (= geometrisches Mittel je Cluster)
    return labels, centers


def kmeans_mit_abweichungskriterium(values, max_k=K_MAX, min_coverage=COVERAGE_MIN,
                                     max_deviation=TOLERANZ_BAND):
    """Erhoeht k schrittweise, bis mindestens `min_coverage` (z.B. 80 %) der Werte
    hoechstens `max_deviation` (z.B. 20 %) relative Abweichung von ihrem
    Cluster-Zentrum haben, oder max_k erreicht ist -- dann wird das beste bis dahin
    gefundene Ergebnis genommen."""
    best = None
    for k in range(1, max_k + 1):
        labels, centers = _kmeans_1d(values, k)
        rel_dev = np.abs(values - centers[labels]) / np.abs(centers[labels])
        share_within = np.mean(rel_dev <= max_deviation)

        if best is None or share_within > best[3]:
            best = (labels, centers, k, share_within)

        if share_within >= min_coverage:
            return labels, centers, k, share_within

        print(f"    (bisher beste Coverage: {best[3]:.0%} bei k={best[2]})")

    return best


def kmeans_aufsteigend_sortiert(values, max_k=K_MAX, min_coverage=COVERAGE_MIN,
                                max_deviation=TOLERANZ_BAND):
    """Duennes Wrapper um kmeans_mit_abweichungskriterium(): sortiert Label und
    Zentren so um, dass Cluster 0 das niedrigste Zentrum hat -- rein kosmetisch
    fuer CSV/Excel/Boxplot. Aendert nichts an der Zuordnung."""
    labels, centers, k, coverage = kmeans_mit_abweichungskriterium(
        values, max_k=max_k, min_coverage=min_coverage, max_deviation=max_deviation
    )
    reihenfolge = np.argsort(centers)
    centers_sortiert = centers[reihenfolge]
    alt_zu_neu = {alt: neu for neu, alt in enumerate(reihenfolge)}
    labels_sortiert = np.array([alt_zu_neu[label] for label in labels])
    return labels_sortiert, centers_sortiert, k, coverage


def clustere_merkmal(werte, merkmal, size_class):
    """Coverage-Clustering fuer EIN Merkmal einer Groessenklasse auf den Werten
    der Trainingsmenge. Werte <= 0 (z.B. Waermebedarf 0 bei unbekanntem
    Sanierungszustand -- np.log braucht > 0) werden nicht mitgeclustert und
    bekommen spaeter direkt 0.0 zugewiesen. Gibt zurueck:
    (labels, zentren, k, coverage) -- labels aligniert mit `werte`, Nullwerte
    behalten Label 0, gehoeren aber zu keinem Cluster (n positive Werte im Fit).

    Das Coverage-Ziel ist pro (size_class, merkmal) nachschlagbar: Standard ist
    COVERAGE_MIN, es sei denn COVERAGE_MIN_AUSNAHMEN (Abschnitt 1) enthaelt fuer
    diese Kombination einen eigenen Wert. Kein Ruecksprung auf lineares KMeans --
    Log-Clustering ist die alleinige Methode."""
    werte = np.asarray(werte, dtype=float)
    ist_positiv = werte > 0
    n_positiv = int(ist_positiv.sum())
    labels = np.zeros(len(werte), dtype=int)

    if n_positiv == 0:
        return labels, np.array([]), 0, 0.0

    if n_positiv < MIN_GEBAEUDE_FUER_CLUSTERING:
        # zu wenige Gebaeude -> nur 1 Cluster: geometrisches Mittel
        zentren = np.array([np.exp(np.mean(np.log(werte[ist_positiv])))])
        rel_abw = np.abs(werte[ist_positiv] - zentren[0]) / zentren[0]
        coverage = float(np.mean(rel_abw <= TOLERANZ_BAND))
        print(f'  {merkmal:<24} nur {n_positiv} Gebaeude -> k=1, '
              f'Coverage {coverage:.0%}')
        return labels, zentren, 1, coverage

    labels[ist_positiv], zentren, k, coverage = kmeans_aufsteigend_sortiert(
        werte[ist_positiv],
        min_coverage=COVERAGE_MIN_AUSNAHMEN.get((size_class, merkmal), COVERAGE_MIN),
    )
    print(f'  {merkmal:<24} k={k}  Coverage {coverage:.0%} '
          f'(Ziel {COVERAGE_MIN_AUSNAHMEN.get((size_class, merkmal), COVERAGE_MIN):.0%})')
    return labels, zentren, k, coverage


print("Log-Raum-KMeans-Hilfsfunktionen definiert (deterministisch).")

Log-Raum-KMeans-Hilfsfunktionen definiert (deterministisch).


### 5.2 Statistik-, Plot- und Bin-Zuordnungsfunktionen

`weise_cluster_anhand_bestehender_bins_zu()` ordnet Werte, die nicht in der
Trainingsmenge lagen (alle übrigen Gebäude, die drei Sanierungsszenarien), in
die bereits gefitteten Cluster ein, statt neue Cluster zu fitten.

In [20]:
def berechne_cluster_statistik(werte, cluster_label, cluster_zentren, k, size_class, merkmal):
    """Tabelle mit einer Zeile je Cluster (n, min, mean, max) fuer genau eine
    (Groessenklasse, Merkmal)-Kombination -- Grundlage fuer CSV-/Excel-Ausgabe
    (Abschnitte 19-20).

    "mean" ist das Cluster-Zentrum aus kmeans_aufsteigend_sortiert()
    (geometrisches Mittel) -- GENAU der Wert, der auch in die
    *_cluster-Ausgabespalten geschrieben wird (Abschnitt 18). "min"/"max"
    bleiben die tatsaechlichen Extremwerte auf der Originalskala."""
    zeilen = []
    for cluster_nr in range(k):
        werte_dieses_clusters = werte[cluster_label == cluster_nr]
        zeilen.append({
            "size_class": size_class, "merkmal": merkmal, "cluster": cluster_nr,
            "n": len(werte_dieses_clusters),
            "min": werte_dieses_clusters.min(),
            "mean": cluster_zentren[cluster_nr],
            "max": werte_dieses_clusters.max(),
        })
    return pd.DataFrame(zeilen)


def speichere_boxplot(werte, cluster_label, cluster_zentren, k, coverage,
                      size_class, merkmal, einheit, unterordner=None):
    """Speichert den Boxplot je Cluster als PDF -- rote Rauten fuer die
    Cluster-Zentren, Titel mit k und erreichter Coverage. Kein separates
    Elbow-/Coverage-Kurven-PDF: dieser eine Plot deckt beides ab."""
    ordner = os.path.join(OUT_DIR, unterordner) if unterordner else os.path.join(OUT_DIR, size_class)
    os.makedirs(ordner, exist_ok=True)

    werte_je_cluster = [werte[cluster_label == c] for c in range(k)]
    beschriftung = [f"Cluster {c + 1}\n(n={len(werte_je_cluster[c]):,})" for c in range(k)]

    abbildung, achse = plt.subplots(figsize=(1.5 * k + 2, 5))
    bp = achse.boxplot(werte_je_cluster, tick_labels=beschriftung,
                       patch_artist=True, showmeans=True)
    for patch in bp["boxes"]:
        patch.set_facecolor("#a6cee3")
        patch.set_alpha(0.7)

    for c, zentrum in enumerate(cluster_zentren):
        achse.scatter(c + 1, zentrum, marker="D", color="red", zorder=3, s=30,
                      label="Cluster-Zentrum" if c == 0 else None)

    achse.set_ylabel(f"{merkmal} [{einheit}]")
    achse.set_title(f"{size_class} -- {merkmal}: k={k}, {coverage:.0%} innerhalb "
                    f"{TOLERANZ_BAND:.0%} Abweichung")
    achse.legend()
    plt.tight_layout()

    pfad = os.path.join(ordner, f"boxplot_{size_class}_{merkmal}.pdf")
    plt.savefig(pfad, bbox_inches="tight")
    plt.close(abbildung)
    return pfad


def weise_cluster_anhand_bestehender_bins_zu(werte, bins):
    """Ordnet jeden Wert einem BEREITS gefitteten Cluster zu (ueber dessen
    [min, max]-Bereich) und gibt dessen Repraesentanten zurueck (Cluster-Zentrum
    ueber die "mean"-Spalte von bins) -- es wird KEIN neues Clustering gefittet.
    Fuer Werte ueber dem hoechsten Bereich (Ausreisser) gilt der Repraesentant des
    Clusters mit dem hoechsten Repraesentanten. Faellt ein Wert in eine Luecke
    zwischen zwei Bereichen, gewinnt der naechstgelegene Cluster-Repraesentant."""
    werte = np.asarray(werte, dtype=float)
    mins = bins["min"].to_numpy()
    maxs = bins["max"].to_numpy()
    means = bins["mean"].to_numpy()

    reihenfolge = np.argsort(maxs)
    mins_sortiert = mins[reihenfolge]
    maxs_sortiert = maxs[reihenfolge]
    means_sortiert = means[reihenfolge]

    index = np.searchsorted(maxs_sortiert, werte, side="left")
    index = np.clip(index, 0, len(maxs_sortiert) - 1)
    ergebnis = means_sortiert[index]

    ist_ausreisser = werte > maxs_sortiert[-1]
    ergebnis[ist_ausreisser] = means.max()

    ist_luecke = (~ist_ausreisser) & (werte < mins_sortiert[index])
    if ist_luecke.any():
        unterschiede = np.abs(means_sortiert[None, :] - werte[ist_luecke, None])
        ergebnis[ist_luecke] = means_sortiert[unterschiede.argmin(axis=1)]

    return ergebnis


print("Statistik-, Plot- und Bin-Zuordnungsfunktionen definiert.")

Statistik-, Plot- und Bin-Zuordnungsfunktionen definiert.


## 6. Clustering je Gebäudetyp — Bedarf

Äußere Schleife: SFH → TH → MFH → AB getrennt. Innere Schleife: Strom und Wärme
unabhängig. Jedes Merkmal bekommt sein eigenes k aus dem Coverage-Kriterium.

**Sonderfall SFH/TH-Strom (bewusst ohne Log-KMeans):** Diese Typen haben
definitionsgemäß immer genau 1 Wohnung, also nur 2 mögliche Werte aus der
CO2online-Tabelle (2 Personen → 2.700 kWh, 3 Personen → 3.500 kWh). Das
Coverage-Kriterium wuerde beide Werte auf **k=1** kollabieren lassen — ihre
Abweichung vom gemeinsamen geometrischen Mittel (~3.074 kWh) betraegt nur ~12-14 %
und liegt damit innerhalb des ±20 %-Bandes. Das aber wuerde die
Personenklassen-Unterscheidung verwerfen. Deshalb bleiben die zwei Werte fest
verdrahtet als k=2 mit dem Wert selbst als Zentrum (Coverage 100 %).

**Hinweis zum PV-Potenzial:** Echte Dachflächen für das SESMG stammen aus dem
Dach-Clustering in Abschnitt 15 (LOD2-Geometrie, ohne angenommene Anteile).

In [21]:
MERKMALE_BEDARF = [
    ("Strombedarf",  "elec_kwh",   "kWh/Jahr"),
    ("Waermebedarf", "heat_total_kwh", "kWh/Jahr"),
]

bedarf_bins       = {}   # (size_class, merkmal) -> Stats-DataFrame (min/mean/max je Cluster)
bedarf_k          = {}   # (size_class, merkmal) -> k
bedarf_labels     = {}   # (size_class, merkmal) -> Labels der Trainingsmenge (fuer Validierung)
alle_statistiken  = []   # Stats-Zeilen aller Clusterings (Bedarf + Dach)
ergebnisse        = []   # Uebersichtszeilen aller Clusterings

for sc in SIZE_CLASSES:
    train_sc = training[training["size_class"] == sc].copy().reset_index(drop=True)
    print(f'\n{"="*60}')
    print(f'  {sc}  ({len(train_sc):,} Gebaeude)')
    print(f'{"="*60}')
    bedarf_k[sc] = {}

    for merkmal, spalte, einheit in MERKMALE_BEDARF:
        werte = train_sc[spalte].values.astype(float)

        # Sonderfall: SFH/TH-Strom fest verdrahtet (siehe Abschnitt 6)
        if sc in ["SFH", "TH"] and spalte == "elec_kwh":
            unique_vals = sorted(train_sc[spalte].unique())
            wert_zu_cluster = {v: i for i, v in enumerate(unique_vals)}
            labels = train_sc[spalte].map(wert_zu_cluster).values
            zentren = np.array(unique_vals)
            k = len(unique_vals)
            coverage = 1.0
            print(f'  {merkmal:<24} fest verdrahtet k={k}  '
                  f'Werte={[int(v) for v in unique_vals]}')
        else:
            labels, zentren, k, coverage = clustere_merkmal(werte, merkmal, sc)

        ist_positiv = werte > 0
        stats = berechne_cluster_statistik(
            werte[ist_positiv], labels[ist_positiv], zentren, k, sc, merkmal
        )
        bedarf_bins[(sc, merkmal)] = stats
        bedarf_k[sc][merkmal] = k
        bedarf_labels[(sc, merkmal)] = labels
        alle_statistiken.append(stats)
        ergebnisse.append({
            "size_class": sc, "merkmal": merkmal, "spalte": spalte,
            "k": k, "coverage": coverage,
            "coverage_ziel": COVERAGE_MIN_AUSNAHMEN.get((sc, merkmal), COVERAGE_MIN),
            "n_gebaeude": int(ist_positiv.sum()),
            "mittelwert": float(werte[ist_positiv].mean()),
        })

        if k > 0:
            speichere_boxplot(werte[ist_positiv], labels[ist_positiv], zentren,
                              k, coverage, sc, merkmal, einheit)

    k_werte = [bedarf_k[sc][m] for m, _, _ in MERKMALE_BEDARF]
    kombis = 1
    for k in k_werte: kombis *= k
    print(f'\n  Kombinationen fuer {sc}: {" x ".join(map(str, k_werte))} = {kombis}')

# Kombinationen je Typ merken fuer Abschnitt 8
bedarf_kombis_je_typ = {}
for sc in SIZE_CLASSES:
    kombis = 1
    for m, _, _ in MERKMALE_BEDARF:
        kombis *= max(bedarf_k[sc][m], 1)
    bedarf_kombis_je_typ[sc] = kombis


  SFH  (2,473,500 Gebaeude)
  Strombedarf              fest verdrahtet k=2  Werte=[2700, 3500]
    (bisher beste Coverage: 30% bei k=1)
    (bisher beste Coverage: 41% bei k=2)
    (bisher beste Coverage: 53% bei k=3)
    (bisher beste Coverage: 67% bei k=4)
  Waermebedarf             k=5  Coverage 81% (Ziel 80%)

  Kombinationen fuer SFH: 2 x 5 = 10

  TH  (490,084 Gebaeude)
  Strombedarf              fest verdrahtet k=2  Werte=[2700, 3500]
    (bisher beste Coverage: 38% bei k=1)
    (bisher beste Coverage: 45% bei k=2)
    (bisher beste Coverage: 59% bei k=3)
    (bisher beste Coverage: 72% bei k=4)
  Waermebedarf             k=5  Coverage 82% (Ziel 80%)

  Kombinationen fuer TH: 2 x 5 = 10

  MFH  (720,288 Gebaeude)
    (bisher beste Coverage: 14% bei k=1)
    (bisher beste Coverage: 37% bei k=2)
    (bisher beste Coverage: 51% bei k=3)
    (bisher beste Coverage: 59% bei k=4)
    (bisher beste Coverage: 59% bei k=4)
    (bisher beste Coverage: 69% bei k=6)
  Strombedarf          

## 7. Gesamtdatensatz-Vergleich

Gleicher Algorithmus, aber alle Gebäudetypen zusammen — zeigt, was beim Mischen
der Typen passiert. Dient nur dem Vergleich, fließt nicht in die Arbeitsdatei ein.

In [22]:
print("GESAMTDATENSATZ — Vergleichsbasis")
print("="*60)

gesamt_bins = {}
gesamt_k = {}
for merkmal, spalte, einheit in MERKMALE_BEDARF:
    werte = training[spalte].values.astype(float)
    print(f'\n  {merkmal}:')
    labels, zentren, k, coverage = clustere_merkmal(werte, merkmal, "GESAMT")
    ist_positiv = werte > 0
    stats = berechne_cluster_statistik(
        werte[ist_positiv], labels[ist_positiv], zentren, k, "GESAMT", merkmal
    )
    gesamt_bins[merkmal] = stats
    gesamt_k[merkmal] = k
    alle_statistiken.append(stats)
    ergebnisse.append({
        "size_class": "GESAMT", "merkmal": merkmal, "spalte": spalte,
        "k": k, "coverage": coverage,
        "coverage_ziel": COVERAGE_MIN,
        "n_gebaeude": int(ist_positiv.sum()),
        "mittelwert": float(werte[ist_positiv].mean()),
    })
    if k > 0:
        speichere_boxplot(werte[ist_positiv], labels[ist_positiv], zentren,
                          k, coverage, "GESAMT", merkmal, einheit, unterordner="gesamt")

gesamt_kombis = 1
for m, _, _ in MERKMALE_BEDARF:
    gesamt_kombis *= max(gesamt_k[m], 1)
print(f'\nGesamtdatensatz: {" x ".join(str(gesamt_k[m]) for m, _, _ in MERKMALE_BEDARF)} = {gesamt_kombis}')

GESAMTDATENSATZ — Vergleichsbasis

  Strombedarf:
  Strombedarf              k=1  Coverage 83% (Ziel 80%)

  Waermebedarf:
    (bisher beste Coverage: 24% bei k=1)
    (bisher beste Coverage: 33% bei k=2)
    (bisher beste Coverage: 44% bei k=3)
    (bisher beste Coverage: 55% bei k=4)
    (bisher beste Coverage: 66% bei k=5)
    (bisher beste Coverage: 78% bei k=6)
  Waermebedarf             k=7  Coverage 86% (Ziel 80%)

Gesamtdatensatz: 1 x 7 = 7


## 8. Kombinationszahlen-Vergleich

- **Gesamtdatensatz**: weniger Kombinationen, aber Archetypen können Gebäudetypen mischen
- **Je Typ**: mehr Kombinationen, aber jeder ist physikalisch sinnvoll

In [23]:
summe_je_typ = sum(bedarf_kombis_je_typ.values())

print("KOMBINATIONSVERGLEICH")
print("="*60)
print(f"\nAnsatz Gesamtdatensatz:")
for merkmal, _, _ in MERKMALE_BEDARF:
    print(f'  {merkmal:<24}: k={gesamt_k[merkmal]}')
print(f'  Gesamt: {" x ".join(str(gesamt_k[m]) for m, _, _ in MERKMALE_BEDARF)} = {gesamt_kombis}')

print(f"\nAnsatz je Gebaeudetyp:")
for sc in SIZE_CLASSES:
    k_werte = [bedarf_k[sc][m] for m, _, _ in MERKMALE_BEDARF]
    print(f'  {sc}: {" x ".join(map(str, k_werte))} = {bedarf_kombis_je_typ[sc]}')
print(f'  Gesamt (Summe): {summe_je_typ}')

KOMBINATIONSVERGLEICH

Ansatz Gesamtdatensatz:
  Strombedarf             : k=1
  Waermebedarf            : k=7
  Gesamt: 1 x 7 = 7

Ansatz je Gebaeudetyp:
  SFH: 2 x 5 = 10
  TH: 2 x 5 = 10
  MFH: 7 x 8 = 56
  AB: 8 x 10 = 80
  Gesamt (Summe): 156


## 9. Abstandsanalyse — Bedarf

Prüft, ob benachbarte Cluster-Zentren sinnvoll verschieden sind. Ein Abstand
unter 20 % würde bedeuten, dass zwei Zentren so dicht liegen, dass sie
potenziell zusammenlegbar wären.

**Einordnung beim Coverage-Kriterium:** Das Coverage-Kriterium garantiert, dass
jedes Gebäude innerhalb ±20 % *seines* Zentrums liegt — es erzwingt nicht, dass
benachbarte Zentren ≥ 20 % auseinanderliegen. Die Analyse dient deshalb als
zusätzliche Kontrolle.

In [24]:
stats_df = pd.concat([bedarf_bins[(sc, m)] for sc in SIZE_CLASSES
                       for m, _, _ in MERKMALE_BEDARF], ignore_index=True)

print("ABSTANDSANALYSE — relative Differenz zwischen benachbarten Cluster-Zentren")
print("="*60)

for sc in SIZE_CLASSES:
    print(f"\n{sc}:")
    for merkmal, _, _ in MERKMALE_BEDARF:
        sc_stats = bedarf_bins[(sc, merkmal)].sort_values("cluster")
        for i in range(len(sc_stats) - 1):
            m0  = sc_stats.iloc[i]["mean"]
            m1v = sc_stats.iloc[i + 1]["mean"]
            abstand = (m1v - m0) / m0 * 100 if m0 > 0 else 0
            flag = "" if abstand >= 20 else "  <- KLEINER ABSTAND"
            print(f'  {merkmal[:20]:<22} Z{i}->Z{i+1}: '
                  f'{m0:>10,.0f} -> {m1v:>10,.0f} kWh/Jahr  ({abstand:.0f}%){flag}')

print("\nAlle Abstaende >= 20% bedeutet: Zentren sind gut getrennt und sinnvoll.")

ABSTANDSANALYSE — relative Differenz zwischen benachbarten Cluster-Zentren

SFH:
  Strombedarf            Z0->Z1:      2,700 ->      3,500 kWh/Jahr  (30%)
  Waermebedarf           Z0->Z1:      5,064 ->     10,020 kWh/Jahr  (98%)
  Waermebedarf           Z1->Z2:     10,020 ->     15,697 kWh/Jahr  (57%)
  Waermebedarf           Z2->Z3:     15,697 ->     24,155 kWh/Jahr  (54%)
  Waermebedarf           Z3->Z4:     24,155 ->     42,080 kWh/Jahr  (74%)

TH:
  Strombedarf            Z0->Z1:      2,700 ->      3,500 kWh/Jahr  (30%)
  Waermebedarf           Z0->Z1:      2,519 ->      5,221 kWh/Jahr  (107%)
  Waermebedarf           Z1->Z2:      5,221 ->      7,969 kWh/Jahr  (53%)
  Waermebedarf           Z2->Z3:      7,969 ->     12,243 kWh/Jahr  (54%)
  Waermebedarf           Z3->Z4:     12,243 ->     22,850 kWh/Jahr  (87%)

MFH:
  Strombedarf            Z0->Z1:      1,900 ->      3,800 kWh/Jahr  (100%)
  Strombedarf            Z1->Z2:      3,800 ->      5,700 kWh/Jahr  (50%)
  Strombedarf     

## 10. Clustergrößen-Validierung

Prüft, ob ein Cluster zu klein für einen verlässlichen Archetyp ist. AB ist der
kritische Fall — nur wenige tausend Gebäude insgesamt, auf mehrere Cluster verteilt.

In [25]:
print("CLUSTERGROESSEN-VALIDIERUNG (Trainingsmenge)")
print("="*60)

for sc in SIZE_CLASSES:
    train_sc = training[training["size_class"] == sc]
    print(f"\n{sc}:")
    for merkmal, _, _ in MERKMALE_BEDARF:
        labels = bedarf_labels[(sc, merkmal)]
        werte = train_sc[[s for m, s, _ in MERKMALE_BEDARF if m == merkmal][0]].values
        groessen = pd.Series(labels[werte > 0]).value_counts().sort_index()
        if groessen.empty:
            continue
        min_n = groessen.min()
        min_c = groessen.idxmin()
        warnung = "  <- WARNUNG: sehr kleiner Cluster" if min_n < 100 else ""
        print(f'  {merkmal[:20]:<22} kleinster Cluster: Z{min_c} mit {min_n:,} Gebaeuden{warnung}')

CLUSTERGROESSEN-VALIDIERUNG (Trainingsmenge)

SFH:
  Strombedarf            kleinster Cluster: Z1 mit 93,988 Gebaeuden
  Waermebedarf           kleinster Cluster: Z0 mit 181,207 Gebaeuden

TH:
  Strombedarf            kleinster Cluster: Z1 mit 53,907 Gebaeuden
  Waermebedarf           kleinster Cluster: Z0 mit 27,664 Gebaeuden

MFH:
  Strombedarf            kleinster Cluster: Z0 mit 74,245 Gebaeuden
  Waermebedarf           kleinster Cluster: Z0 mit 25,506 Gebaeuden

AB:
  Strombedarf            kleinster Cluster: Z7 mit 301 Gebaeuden
  Waermebedarf           kleinster Cluster: Z0 mit 105 Gebaeuden


## 11. Sanierungszustand-Kreuztabelle

Prüft, ob die Wärme-Cluster Gebäude implizit nach Sanierungszustand trennen.

- **Hoher Bedarf = überwiegend Zustand 1 und niedriger Bedarf = überwiegend Zustand 3** → das Clustering erfasst den Sanierungsgrad implizit
- **Jeder Cluster hat gleiche Mischung** → das Clustering erfasst nur die Größe, nicht die Sanierung

In [26]:
print("SANIERUNGSZUSTAND-KREUZTABELLE")
print("="*60)

heat_merkmal = "Waermebedarf"
spalten = [s for m, s, _ in MERKMALE_BEDARF if m == heat_merkmal]
heat_spalte = spalten[0] if spalten else "heat_total_kwh"

for sc in SIZE_CLASSES:
    train_sc = training[training["size_class"] == sc].copy().reset_index(drop=True)
    labels = bedarf_labels[(sc, heat_merkmal)]
    werte = train_sc[heat_spalte].values
    frame = pd.DataFrame({
        "cluster": labels,
        "refurbishment_state": train_sc["refurbishment_state"].values,
    })
    frame = frame[werte > 0]
    if frame.empty:
        continue
    ct = pd.crosstab(frame["cluster"], frame["refurbishment_state"],
                     normalize="index") * 100
    ct.index.name = "Waerme-Cluster"
    ct.columns.name = "Sanierungszustand"
    print(f"\n{sc}:")
    print(ct.round(1).to_string())

SANIERUNGSZUSTAND-KREUZTABELLE

SFH:
Sanierungszustand     1     2     3
Waerme-Cluster                     
0                  11.8  52.6  35.6
1                  20.6  74.0   5.4
2                  34.8  64.5   0.7
3                  56.3  43.5   0.2
4                  71.4  28.6   0.0

TH:
Sanierungszustand     1     2     3
Waerme-Cluster                     
0                   2.4  38.8  58.8
1                  14.6  82.1   3.3
2                  32.9  67.0   0.2
3                  68.0  31.9   0.1
4                  78.5  21.4   0.1

MFH:
Sanierungszustand     1     2     3
Waerme-Cluster                     
0                  10.1  56.4  33.5
1                  17.3  65.9  16.8
2                  24.6  68.0   7.5
3                  29.3  68.8   1.9
4                  34.1  65.4   0.5
5                  50.3  49.6   0.2
6                  70.1  29.9   0.0
7                  76.8  23.2   0.0

AB:
Sanierungszustand     1     2     3
Waerme-Cluster                     
0          

## 12. Dachdaten — LOD2 laden und per Gebäude-ID joinen

Der LOD2-Datensatz (Level of Detail 2) enthält 3D-Dachgeometrie aus dem
NRW-Gebäudebestandsscan.

Verwendete Spalten:
- `avg_tilt_deg` — mittlerer Neigungswinkel je Gebäude (Flachdach-Erkennung)
- `area_south_m2` — südwärtige Dachfläche
- `area_east_m2` / `area_west_m2` — ost- und westwärtige Flächen
- `area_total_m2` — Gesamtdachfläche (für Flachdächer)

Nordwärtige Fläche wird durchgängig ausgeschlossen.

Die Arbeitsdatei hat `DEA_` vor jeder Gebäude-ID vorangestellt, der
Dachdatensatz nicht (`DEA_DENW01AL1000CrNT` vs. `DENW01AL1000CrNT`) — `DEA_`
streichen erlaubt direkte Zuordnung, ~99,3 % Treffer.

**Left Join, kein Inner Join** — nicht zugeordnete Gebäude bleiben mit
`roof_category = 'unknown'` in der Population (Abschnitt 13) statt still
verschwinden. Die Arbeitsdatei muss alle Gebäude abdecken.

In [27]:
roofs = gpd.read_parquet(FILE_ROOF)
# height_m (Dach-Hoehe) und roof_type_osm werden im Weiteren nicht benoetigt
# -- und height_m wuerde mit der DEA-Gebaeudehoehe kollidieren.
roofs = roofs.drop(columns=[
    c for c in ["tile", "roof_type_code", "function",
                "gemeinde", "creation_date", "geometry",
                "height_m", "roof_type_osm"]
    if c in roofs.columns
])
print(f'Dachdatensatz geladen: {len(roofs):,} Gebaeude')

gebaeude["building_id"] = gebaeude["id"].str.replace("DEA_", "", regex=False)

# Idempotenz: Ein frueherer Lauf hat die Dachspalten in die Arbeitsdatei
# zurueckgeschrieben (height_m damals sogar als Suffix-Paar height_m_x /
# height_m_y, weil der Merge kollidierte). Ohne Bereinigung wuerde dieser
# Merge gleichnamige Spalten mit _x/_y-Suffixen versehen und area_east_m2
# & Co. waeren nicht mehr ansprechbar. height_m_x ist die DEA-Gebaeudehoehe
# -- sie wird zu height_m zurueckbenannt, height_m_y (Dach-Hoehe) verworfen.
if "height_m_x" in gebaeude.columns and "height_m" not in gebaeude.columns:
    gebaeude = gebaeude.rename(columns={"height_m_x": "height_m"})
gebaeude = gebaeude.drop(columns=[
    c for c in ["height_m_x", "height_m_y", "roof_type_osm"]
    if c in gebaeude.columns
])
kollidierende_dachspalten = [c for c in roofs.columns
                            if c != "building_id" and c in gebaeude.columns]
if kollidierende_dachspalten:
    print(f'Vor dem Merge entfernt (Reste frueherer Laeufe): '
          f'{kollidierende_dachspalten}')
    gebaeude = gebaeude.drop(columns=kollidierende_dachspalten)

# Left Join — jedes Gebaeude bleibt, nicht zugeordnete bekommen NaN-Dachspalten
merged = gebaeude.merge(roofs, on="building_id", how="left", indicator=True)
n_matched = (merged["_merge"] == "both").sum()
merged = merged.drop(columns="_merge")

print(f'Gebaeude (Arbeitsdatei) : {len(gebaeude):,}')
print(f'Dachdatensatz           : {len(roofs):,}')
print(f'Zugeordnet              : {n_matched:,}  ({n_matched/len(gebaeude)*100:.1f}%)')
print(f'Nicht zugeordnet        : {len(gebaeude)-n_matched:,}  '
      f"(keine Dachdaten — roof_category='unknown')")

Dachdatensatz geladen: 5,097,348 Gebaeude
Gebaeude (Arbeitsdatei) : 4,134,282
Dachdatensatz           : 5,097,348
Zugeordnet              : 4,104,414  (99.3%)
Nicht zugeordnet        : 29,868  (keine Dachdaten — roof_category='unknown')


## 13. Dach-Klassifikation

Drei Kategorien anhand des mittleren Neigungswinkels:
- **Flachdach** (`avg_tilt_deg < 10`) — PV-Clustering auf `area_total_m2`
- **Satteldach** (`avg_tilt_deg >= 10`) — Süd und Ost-West getrennt clustern
- **Unbekannt** (Neigung NaN, d. h. kein LOD2-Treffer) — vom Dach-Clustering und von der Zuweisung ausgeschlossen, Dach-Cluster bleiben 0

Ost und West werden zu einer Ost-West-Variable zusammengefasst, weil:
- ein symmetrisches Satteldach ungefähr gleiche Ost- und West-Flächen hat
- die Zusammenfassung Cluster-Variablen reduziert, ohne physikalische Bedeutung zu verlieren

Gebäude mit nur nordwärtiger Fläche bekommen `has_pv_area = False` — vom Clustering ausgeschlossen.

In [28]:
# Ost-West kombiniert (Nord ausgeschlossen)
merged["area_eastwest_m2"] = merged["area_east_m2"] + merged["area_west_m2"]

# Dachtyp klassifizieren
merged["roof_category"] = "gabled"
merged.loc[merged["avg_tilt_deg"] < FLAT_TILT_THRESHOLD, "roof_category"] = "flat"
merged.loc[merged["avg_tilt_deg"].isna(), "roof_category"] = "unknown"

# Gebaeude ohne nutzbare PV-Flaeche markieren (nur Nord, oder unbekannt)
merged["has_pv_area"] = (
    (merged["area_south_m2"] > 0) | (merged["area_eastwest_m2"] > 0)
)

print("Dachkategorien:")
for cat, count in merged["roof_category"].value_counts().items():
    print(f'  {cat:<10}: {count:>10,}  ({count/len(merged)*100:.1f}%)')

ohne_pv = (~merged["has_pv_area"]).sum()
print(f'\nGebaeude ohne nutzbare PV-Flaeche: {ohne_pv:,} '
      f'({(~merged["has_pv_area"]).mean()*100:.1f}%)')

print("\nDachkategorie je Gebaeudetyp:")
print(pd.crosstab(merged["size_class"], merged["roof_category"]).to_string())

Dachkategorien:
  gabled    :  3,778,806  (91.4%)
  flat      :    251,807  (6.1%)
  unknown   :    103,669  (2.5%)

Gebaeude ohne nutzbare PV-Flaeche: 140,275 (3.4%)

Dachkategorie je Gebaeudetyp:
roof_category    flat   gabled  unknown
size_class                             
AB               1656     2503     1190
MFH             87021   780300    30096
SFH            112739  2552913    58148
TH              50391   443090    14235


## 14. Dachflächen-Verteilungen

Süd- und Ost-West-Flächenverteilungen vor dem Clustering — zeigen, warum
Clustering nötig ist: breite Streuung über alle Gebäudetypen.

In [29]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, sc in zip(axes.flat, SIZE_CLASSES):
    subset = merged[merged["size_class"] == sc]
    gabled = subset[subset["roof_category"] != "flat"]

    ax.hist(gabled["area_south_m2"].clip(upper=gabled["area_south_m2"].quantile(0.99)),
            bins=50, alpha=0.7, color="goldenrod", label="Suedflaeche")
    ax.hist(gabled["area_eastwest_m2"].clip(upper=gabled["area_eastwest_m2"].quantile(0.99)),
            bins=50, alpha=0.7, color="steelblue", label="Ost-West-Flaeche")
    ax.set_title(f"{sc} — nur Satteldaecher", fontsize=11, fontweight="bold")
    ax.set_xlabel("Dachflaeche (m2)")
    ax.set_ylabel("Anzahl")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Dachflaechenverteilungen je Gebaeudetyp (gekappt bei P99)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "dachflaechenverteilungen.pdf"), bbox_inches="tight")
plt.close()

## 15. Clustering je Gebäudetyp — Dach

Wiederverwendung der Log-Raum-KMeans-Funktionen aus Abschnitt 5 — gleiche
Methode wie beim Bedarf, nur auf Dachflächen angewendet.

**Satteldach-Gebäude mit nutzbarer PV-Fläche:**
- `area_south_m2` und `area_eastwest_m2` werden unabhängig geclustert
- **Nullwert-Ausschluss:** Ein Satteldach-Gebäude, dessen jeweilige Richtung *nicht* dominant ist, hat dort den Wert 0 — das ist kein kleiner Messwert, sondern heißt „diese Richtung existiert nicht". Diese Werte werden vor dem Clustering herausgefiltert und bekommen fest 0.0 zugewiesen. So entstehen keine künstlichen „Nullen-Cluster", die kleine, aber echte Flächen nach unten ziehen würden.

**Flachdächer mit nutzbarer PV-Fläche:**
- Clustering auf `area_total_m2` — Ausrichtung ist bei Flachdächern nicht anwendbar

**Gebäude ohne PV-Fläche (nur Nord) / unbekannt:**
- bekommen 0.0 in beiden Dach-Cluster-Spalten

In [30]:
DAVARIABLEN = [
    ("Sueddachflaeche",    "area_south_m2",    "m2"),
    ("OstWestdachflaeche", "area_eastwest_m2", "m2"),
]

dach_bins       = {}   # (key, merkmal) -> Stats-DataFrame
dach_k          = {}   # sc -> {"gabled_sued": k, "gabled_ostwest": k, "flach": k}
dach_kombis_je_typ = {}

for sc in SIZE_CLASSES:
    sc_all    = merged[merged["size_class"] == sc].copy().reset_index(drop=True)
    sc_gabled = sc_all[sc_all["roof_category"] != "flat"].copy().reset_index(drop=True)
    sc_flat   = sc_all[sc_all["roof_category"] == "flat"].copy().reset_index(drop=True)

    print(f'\n{"="*60}')
    print(f'  {sc}  ({len(sc_all):,} Gebaeude insgesamt)')
    print(f'  Satteldach: {len(sc_gabled):,}  |  Flachdach: {len(sc_flat):,}  |  '
          f'Ohne PV-Flaeche: {(~sc_all["has_pv_area"]).sum():,}')
    print(f'{"="*60}')

    dach_k[sc] = {}

    # -- Satteldach-Gebaeude mit nutzbarer PV-Flaeche (Nullwert-Ausschluss) --
    gabled_pv = sc_gabled[sc_gabled["has_pv_area"]].copy().reset_index(drop=True)
    print(f'  Clustering Satteldaecher mit nutzbarer PV-Flaeche: {len(gabled_pv):,}')

    for merkmal, spalte, einheit in DAVARIABLEN:
        werte = gabled_pv[spalte].values.astype(float)
        ist_null = werte <= 0
        if ist_null.any():
            print(f'  {int(ist_null.sum()):,} Gebaeude ohne Flaeche in dieser Richtung '
                  "-> direkt 0.0, nicht mitgeclustert")
        labels, zentren, k, coverage = clustere_merkmal(werte, merkmal, sc)
        ist_positiv = werte > 0
        stats = berechne_cluster_statistik(
            werte[ist_positiv], labels[ist_positiv], zentren, k, sc, merkmal
        )
        dach_bins[(sc, merkmal)] = stats
        dach_k[sc][f"gabled_{spalte}"] = k
        alle_statistiken.append(stats)
        ergebnisse.append({
            "size_class": sc, "merkmal": merkmal, "spalte": spalte,
            "k": k, "coverage": coverage,
            "coverage_ziel": COVERAGE_MIN_AUSNAHMEN.get((sc, merkmal), COVERAGE_MIN),
            "n_gebaeude": int(ist_positiv.sum()),
            "mittelwert": float(werte[ist_positiv].mean()) if ist_positiv.any() else 0.0,
        })
        if k > 0:
            speichere_boxplot(werte[ist_positiv], labels[ist_positiv], zentren,
                              k, coverage, sc, merkmal, einheit)

    # -- Flachdaecher --
    flat_pv = sc_flat[sc_flat["has_pv_area"]].copy().reset_index(drop=True)
    if len(flat_pv) >= MIN_GEBAEUDE_FUER_CLUSTERING:
        print(f'  Clustering Flachdaecher: {len(flat_pv):,}')
        werte = flat_pv["area_total_m2"].values.astype(float)
        merkmal, einheit = "Flachdachflaeche", "m2"
        labels, zentren, k, coverage = clustere_merkmal(werte, merkmal, sc)
        ist_positiv = werte > 0
        stats = berechne_cluster_statistik(
            werte[ist_positiv], labels[ist_positiv], zentren, k, sc, merkmal
        )
        dach_bins[(sc, merkmal)] = stats
        dach_k[sc]["flat_total"] = k
        alle_statistiken.append(stats)
        ergebnisse.append({
            "size_class": sc, "merkmal": merkmal, "spalte": "area_total_m2",
            "k": k, "coverage": coverage,
            "coverage_ziel": COVERAGE_MIN_AUSNAHMEN.get((sc, merkmal), COVERAGE_MIN),
            "n_gebaeude": int(ist_positiv.sum()),
            "mittelwert": float(werte[ist_positiv].mean()) if ist_positiv.any() else 0.0,
        })
        if k > 0:
            speichere_boxplot(werte[ist_positiv], labels[ist_positiv], zentren,
                              k, coverage, sc, merkmal, einheit)
    else:
        print(f'  Flachdaecher zu wenige zum Clustern ({len(flat_pv)}) -> kein Cluster')
        dach_k[sc]["flat_total"] = 0

    # -- Kombinationen --
    k_s = dach_k[sc].get("gabled_area_south_m2", 0)
    k_e = dach_k[sc].get("gabled_area_eastwest_m2", 0)
    k_f = dach_k[sc].get("flat_total", 0)
    kombi = k_s * k_e + k_f
    dach_kombis_je_typ[sc] = kombi
    print(f'\n  Satteldach-Kombinationen : {k_s} x {k_e} = {k_s*k_e}')
    print(f'  Flachdach-Kombinationen  : {k_f}')
    print(f'  Gesamt fuer {sc}          : {kombi}')


  SFH  (2,723,800 Gebaeude insgesamt)
  Satteldach: 2,611,061  |  Flachdach: 112,739  |  Ohne PV-Flaeche: 78,245
  Clustering Satteldaecher mit nutzbarer PV-Flaeche: 2,541,618
  963,354 Gebaeude ohne Flaeche in dieser Richtung -> direkt 0.0, nicht mitgeclustert
    (bisher beste Coverage: 31% bei k=1)
    (bisher beste Coverage: 39% bei k=2)
    (bisher beste Coverage: 49% bei k=3)
    (bisher beste Coverage: 59% bei k=4)
    (bisher beste Coverage: 72% bei k=5)
    (bisher beste Coverage: 77% bei k=6)
  Sueddachflaeche          k=7  Coverage 84% (Ziel 80%)
  1,018,681 Gebaeude ohne Flaeche in dieser Richtung -> direkt 0.0, nicht mitgeclustert
    (bisher beste Coverage: 26% bei k=1)
    (bisher beste Coverage: 36% bei k=2)
    (bisher beste Coverage: 44% bei k=3)
    (bisher beste Coverage: 53% bei k=4)
    (bisher beste Coverage: 61% bei k=5)
    (bisher beste Coverage: 73% bei k=6)
  OstWestdachflaeche       k=7  Coverage 80% (Ziel 80%)
  Clustering Flachdaecher: 103,937
    (bishe

## 16. Dach-Kombinationen

Dach-Kombinationen sind anders strukturiert als Bedarfs-Kombinationen:
- **Satteldach**: Süd-Cluster × Ost-West-Cluster (zwei unabhängige Dimensionen)
- **Flachdach**: eigene Cluster auf Gesamtdfläche (keine Ausrichtungs-Dimension)

Diese multiplizieren sich mit den Bedarfs-Kombinationen zur Gesamt-Archetypenzahl.

In [31]:
dach_gesamt = sum(dach_kombis_je_typ.values())
bedarf_gesamt = sum(bedarf_kombis_je_typ.values())

print("KOMBINATIONEN — DACH")
print("="*60)
print(f"\n  Dachflaechen-Kombinationen je Gebaeudetyp:")
for sc in SIZE_CLASSES:
    k_s = dach_k[sc].get("gabled_area_south_m2", 0)
    k_e = dach_k[sc].get("gabled_area_eastwest_m2", 0)
    k_f = dach_k[sc].get("flat_total", 0)
    print(f'  {sc}: Satteldach {k_s}x{k_e}={k_s*k_e}  +  Flachdach {k_f}  =  {dach_kombis_je_typ[sc]}')

print(f'\n  Dach-Kombinationen gesamt: {dach_gesamt}')
print(f'\n  Kombiniert mit dem Bedarfs-Clustering:')
print(f'    Bedarf ({bedarf_gesamt} Kombinationen) x {dach_gesamt} = '
      f'{bedarf_gesamt*dach_gesamt:,} Archetypen insgesamt')

KOMBINATIONEN — DACH

  Dachflaechen-Kombinationen je Gebaeudetyp:
  SFH: Satteldach 7x7=49  +  Flachdach 7  =  56
  TH: Satteldach 6x5=30  +  Flachdach 5  =  35
  MFH: Satteldach 8x9=72  +  Flachdach 8  =  80
  AB: Satteldach 11x11=121  +  Flachdach 9  =  130

  Dach-Kombinationen gesamt: 301

  Kombiniert mit dem Bedarfs-Clustering:
    Bedarf (156 Kombinationen) x 301 = 46,956 Archetypen insgesamt


## 17. Dach-Abstandsanalyse

Wie Abschnitt 9, für die Dach-Cluster: relative Differenz zwischen benachbarten
Zentren, Mindestschwelle 20 %.

**Hinweis:** Dank des Nullwert-Ausschlusses aus Abschnitt 15 enthält jeder
Cluster nur echte Flächen > 0. Gebäude ohne Fläche in einer Richtung bekommen
fest 0.0 statt eines Cluster-Repräsentanten.

In [32]:
print("ABSTANDSANALYSE DACH — relative Differenz zwischen benachbarten Zentren")
print("="*60)

for sc in SIZE_CLASSES:
    print(f"\n  {sc}:")
    for merkmal, _, _ in DAVARIABLEN:
        sc_stats = dach_bins.get((sc, merkmal))
        if sc_stats is None or sc_stats.empty:
            continue
        sc_stats = sc_stats.sort_values("cluster")
        for i in range(len(sc_stats) - 1):
            m0  = sc_stats.iloc[i]["mean"]
            m1v = sc_stats.iloc[i + 1]["mean"]
            abstand = (m1v - m0) / m0 * 100 if m0 > 0 else 0
            flag = "" if abstand >= 20 else "  <- KLEINER ABSTAND"
            print(f'  {merkmal[:22]:<24} Z{i}->Z{i+1}: '
                  f'{m0:>8,.1f} -> {m1v:>8,.1f} m2  ({abstand:.0f}%){flag}')

    sc_stats = dach_bins.get((sc, "Flachdachflaeche"))
    if sc_stats is not None and not sc_stats.empty:
        sc_stats = sc_stats.sort_values("cluster")
        for i in range(len(sc_stats) - 1):
            m0  = sc_stats.iloc[i]["mean"]
            m1v = sc_stats.iloc[i + 1]["mean"]
            abstand = (m1v - m0) / m0 * 100 if m0 > 0 else 0
            flag = "" if abstand >= 20 else "  <- KLEINER ABSTAND"
            print(f'  {"Flachdachflaeche":<24} Z{i}->Z{i+1}: '
                  f'{m0:>8,.1f} -> {m1v:>8,.1f} m2  ({abstand:.0f}%){flag}')

ABSTANDSANALYSE DACH — relative Differenz zwischen benachbarten Zentren

  SFH:
  Sueddachflaeche          Z0->Z1:      1.3 ->      6.6 m2  (420%)
  Sueddachflaeche          Z1->Z2:      6.6 ->     19.7 m2  (196%)
  Sueddachflaeche          Z2->Z3:     19.7 ->     37.6 m2  (91%)
  Sueddachflaeche          Z3->Z4:     37.6 ->     56.7 m2  (51%)
  Sueddachflaeche          Z4->Z5:     56.7 ->     81.8 m2  (44%)
  Sueddachflaeche          Z5->Z6:     81.8 ->    139.0 m2  (70%)
  OstWestdachflaeche       Z0->Z1:      3.5 ->     14.4 m2  (307%)
  OstWestdachflaeche       Z1->Z2:     14.4 ->     36.5 m2  (153%)
  OstWestdachflaeche       Z2->Z3:     36.5 ->     72.1 m2  (97%)
  OstWestdachflaeche       Z3->Z4:     72.1 ->    111.2 m2  (54%)
  OstWestdachflaeche       Z4->Z5:    111.2 ->    163.7 m2  (47%)
  OstWestdachflaeche       Z5->Z6:    163.7 ->    291.8 m2  (78%)
  Flachdachflaeche         Z0->Z1:     10.9 ->     40.7 m2  (272%)
  Flachdachflaeche         Z1->Z2:     40.7 ->     64.9 m

## 18. Cluster-Repräsentanten allen Gebäuden zuweisen

Jedes Gebäude — nicht nur die Trainingsmengen aus Abschnitten 3 und 15 — bekommt
über `weise_cluster_anhand_bestehender_bins_zu()` den Repräsentanten seines
nächsten Clusters. Das ist, was `heat_cluster_1/2/3` und `elec_cluster` an
`03_nrw_spitzenwaermeleistung.ipynb` weitergeben (die Spitzenleistung soll den
Archetyp widerspiegeln, nicht den exakten individuellen Bedarf), und was
`roof_cluster_eastwest` / `roof_cluster_south` für die PV-Dimensionierung des
SESMG liefern.

`heat_cluster_1/2/3` entsprechen den drei Sanierungsszenarien
(`heat_demand_1/2/3`) — dieselben, auf `heat_total_kwh` (dem *tatsächlichen*
Sanierungszustand) gefitteten Cluster werden für alle drei wiederverwendet, da
Größe und Typ eines Gebäudes über Szenarien hinweg konstant bleiben, nur der
Sanierungszustand nicht.

Nullwert-Behandlung (übernommen aus dem NL-Repositorium):
- Wärme-Szenarien/Strom: Werte werden über die Bins zugeordnet (auch 0 — altes
  Verhalten, 0 landet beim niedrigsten Cluster-Repräsentanten),
- Dach: Werte ≤ 0 oder NaN bekommen fest 0.0 (kein „kleinster Cluster" für
  „Fläche existiert nicht"), Flachdach-Bins fehlen → 0.0,
- `roof_category == 'unknown'` (kein LOD2-Treffer) bleibt 0 in beiden
  Dach-Cluster-Spalten.

In [33]:
zugeordnet = merged.copy()

zugeordnet["heat_cluster_1"] = 0.0
zugeordnet["heat_cluster_2"] = 0.0
zugeordnet["heat_cluster_3"] = 0.0
zugeordnet["elec_cluster"] = 0.0
zugeordnet["roof_cluster_eastwest"] = 0.0
zugeordnet["roof_cluster_south"] = 0.0

def _bins_oder_leer(bins):
    return bins if bins is not None and not bins.empty else None

for sc in SIZE_CLASSES:
    sc_mask = zugeordnet["size_class"] == sc
    if not sc_mask.any():
        continue

    heat_bins = _bins_oder_leer(bedarf_bins.get((sc, "Waermebedarf")))
    if heat_bins is not None:
        zugeordnet.loc[sc_mask, "heat_cluster_1"] = weise_cluster_anhand_bestehender_bins_zu(
            zugeordnet.loc[sc_mask, "heat_demand_1"], heat_bins)
        zugeordnet.loc[sc_mask, "heat_cluster_2"] = weise_cluster_anhand_bestehender_bins_zu(
            zugeordnet.loc[sc_mask, "heat_demand_2"], heat_bins)
        zugeordnet.loc[sc_mask, "heat_cluster_3"] = weise_cluster_anhand_bestehender_bins_zu(
            zugeordnet.loc[sc_mask, "heat_demand_3"], heat_bins)

    elec_bins = _bins_oder_leer(bedarf_bins.get((sc, "Strombedarf")))
    if elec_bins is not None:
        zugeordnet.loc[sc_mask, "elec_cluster"] = weise_cluster_anhand_bestehender_bins_zu(
            zugeordnet.loc[sc_mask, "elec_kwh"], elec_bins)

    # Satteldaecher: Ost-West und Sued getrennt, Nullwerte fest 0.0
    gabled_mask = sc_mask & (zugeordnet["roof_category"] == "gabled")
    if gabled_mask.any():
        ew_bins = _bins_oder_leer(dach_bins.get((sc, "OstWestdachflaeche")))
        if ew_bins is not None:
            werte_ew = zugeordnet.loc[gabled_mask, "area_eastwest_m2"].fillna(0.0).values
            erg = weise_cluster_anhand_bestehender_bins_zu(werte_ew, ew_bins)
            erg[werte_ew <= 0] = 0.0
            zugeordnet.loc[gabled_mask, "roof_cluster_eastwest"] = erg

        s_bins = _bins_oder_leer(dach_bins.get((sc, "Sueddachflaeche")))
        if s_bins is not None:
            werte_s = zugeordnet.loc[gabled_mask, "area_south_m2"].fillna(0.0).values
            erg = weise_cluster_anhand_bestehender_bins_zu(werte_s, s_bins)
            erg[werte_s <= 0] = 0.0
            zugeordnet.loc[gabled_mask, "roof_cluster_south"] = erg

    # Flachdaecher: kombinierte Flaeche in roof_cluster_south, Ost-West bleibt 0
    flat_mask = sc_mask & (zugeordnet["roof_category"] == "flat") & zugeordnet["has_pv_area"]
    if flat_mask.any():
        flat_bins = _bins_oder_leer(dach_bins.get((sc, "Flachdachflaeche")))
        if flat_bins is not None:
            flat_area = (
                zugeordnet.loc[flat_mask, "area_eastwest_m2"].fillna(0.0)
                + zugeordnet.loc[flat_mask, "area_south_m2"].fillna(0.0)
                + zugeordnet.loc[flat_mask, "area_north_m2"].fillna(0.0)
            ).values
            erg = weise_cluster_anhand_bestehender_bins_zu(flat_area, flat_bins)
            erg[flat_area <= 0] = 0.0
            zugeordnet.loc[flat_mask, "roof_cluster_south"] = erg

print("Cluster-Repraesentanten zugeordnet.")
print(zugeordnet[["heat_cluster_1", "heat_cluster_2", "heat_cluster_3",
                  "elec_cluster", "roof_cluster_eastwest",
                  "roof_cluster_south"]].describe())

Cluster-Repraesentanten zugeordnet.
       heat_cluster_1  heat_cluster_2  heat_cluster_3  elec_cluster  \
count    4.134282e+06    4.134282e+06    4.134282e+06  4.134282e+06   
mean     2.767703e+04    1.887646e+04    7.773198e+03  4.320180e+03   
std      2.202366e+04    1.544170e+04    6.372155e+03  4.643567e+03   
min      2.518853e+03    2.518853e+03    2.518853e+03  1.900000e+03   
25%      1.569721e+04    1.002020e+04    5.064321e+03  2.700000e+03   
50%      2.415533e+04    1.569721e+04    5.064321e+03  2.700000e+03   
75%      4.208027e+04    2.415533e+04    1.002020e+04  2.700000e+03   
max      6.096547e+05    6.096547e+05    6.096547e+05  1.563310e+05   

       roof_cluster_eastwest  roof_cluster_south  
count           4.134282e+06        4.134282e+06  
mean            6.738524e+01        4.182378e+01  
std             8.373833e+01        5.184129e+01  
min             0.000000e+00        0.000000e+00  
25%             0.000000e+00        0.000000e+00  
50%             3.

## 19. In die gemeinsame Arbeitsdatei schreiben

Deckt jedes Gebäude aus `nrw_buildings.parquet` ab (nicht nur die Trainingsmengen),
sodass alle Folge-Notebooks eine einfache Zuordnung über `id` machen können,
ohne Zeilen zu verlieren. Die Cluster-Repräsentanten (geometrische Mittel) werden
auf zwei Nachkommastellen gerundet — volle Fließkomma-Präzision bringt keinen
Mehrwert und vergrößert nur die Datei (wie im NL-Repositorium).

In [34]:
# Zuweisungs-Spalten zurueck in das Arbeits-DataFrame uebernehmen
cluster_spalten = ["heat_cluster_1", "heat_cluster_2", "heat_cluster_3",
                   "elec_cluster", "roof_cluster_eastwest", "roof_cluster_south"]
for spalte in cluster_spalten:
    merged[spalte] = zugeordnet[spalte]
merged[cluster_spalten] = merged[cluster_spalten].fillna(0.0).round(2)

# CSV-Zuordnungen je Groessenklasse speichern (NL-Benennung)
DEMAND_SPALTEN = ["id", "size_class", "elec_kwh", "heat_total_kwh",
                  "elec_cluster", "heat_cluster_1", "heat_cluster_2", "heat_cluster_3"]
ROOF_SPALTEN   = ["building_id", "size_class", "roof_category", "has_pv_area",
                  "avg_tilt_deg", "area_south_m2", "area_eastwest_m2",
                  "area_total_m2", "area_north_m2",
                  "roof_cluster_south", "roof_cluster_eastwest"]

for sc in SIZE_CLASSES:
    ordner = os.path.join(OUT_DIR, sc)
    os.makedirs(ordner, exist_ok=True)
    teilmenge = merged[merged["size_class"] == sc]

    demand_pfad = os.path.join(ordner, f"Demand_{sc}_assignments.csv")
    teilmenge[[c for c in DEMAND_SPALTEN if c in teilmenge.columns]].to_csv(
        demand_pfad, index=False)

    roof_pfad = os.path.join(ordner, f"Roof_{sc}_assignments.csv")
    teilmenge[[c for c in ROOF_SPALTEN if c in teilmenge.columns]].to_csv(
        roof_pfad, index=False)
    print(f'{sc:5} gespeichert: {demand_pfad}')
    print(f'      gespeichert: {roof_pfad}')

# Arbeitsdatei ueberschreiben — alle urspruenglichen Spalten bleiben erhalten.
# Die centroide-Spalte aus Abschnitt 2 ist nur eine Arbeitshilfe (Shapely-Punkte
# als Objekt-Spalte) und wird vor dem Schreiben wieder entfernt; 03-05 berechnen
# sie bei Bedarf aus der footprint-Geometrie neu.
merged = merged.drop(columns=[c for c in ["centroid"] if c in merged.columns])
merged.to_parquet(FILE_GEBAEUDE, index=False)
print(f'\nueberschrieben: {FILE_GEBAEUDE}  ({len(merged):,} Gebaeude, '
      f'{len(merged.columns)} Spalten)')

SFH   gespeichert: ./nrw_results/SFH/Demand_SFH_assignments.csv
      gespeichert: ./nrw_results/SFH/Roof_SFH_assignments.csv
TH    gespeichert: ./nrw_results/TH/Demand_TH_assignments.csv
      gespeichert: ./nrw_results/TH/Roof_TH_assignments.csv
MFH   gespeichert: ./nrw_results/MFH/Demand_MFH_assignments.csv
      gespeichert: ./nrw_results/MFH/Roof_MFH_assignments.csv
AB    gespeichert: ./nrw_results/AB/Demand_AB_assignments.csv
      gespeichert: ./nrw_results/AB/Roof_AB_assignments.csv

ueberschrieben: ./nrw_buildings.parquet  (4,134,282 Gebaeude, 36 Spalten)


## 20. Excel-Zusammenfassung speichern

Eine einzige `Clustering_summary.xlsx` mit den NL-Sheet-Namen: `Uebersicht`
(eine Zeile je (Großenklasse, Merkmal) mit k und Coverage), `Kombinationen`
(Bedarf und Dach) sowie `Stats_{SFH,TH,MFH,AB}` und `Alle_Stats`.

In [35]:
ergebnis_uebersicht = pd.DataFrame(ergebnisse)
alle_stats_df = pd.concat(alle_statistiken, ignore_index=True)

# Zwei Nachkommastellen genuegen fuer Flaechen-/Bedarfsgroessen; Coverage
# ist ein Anteil -- drei Nachkommastellen sind dort aussagekraeftiger.
ergebnis_uebersicht["mittelwert"] = ergebnis_uebersicht["mittelwert"].round(2)
ergebnis_uebersicht[["coverage", "coverage_ziel"]] = (
    ergebnis_uebersicht[["coverage", "coverage_ziel"]].round(3))
alle_stats_df[["min", "mean", "max"]] = alle_stats_df[["min", "mean", "max"]].round(2)

kombinations_zeilen = []
for sc in SIZE_CLASSES:
    kombinations_zeilen.append({
        "Groessenklasse": sc,
        "k Strom":  bedarf_k[sc].get("Strombedarf", 0),
        "k Waerme": bedarf_k[sc].get("Waermebedarf", 0),
        "Bedarf-Kombis": bedarf_kombis_je_typ[sc],
        "k Sued":   dach_k[sc].get("gabled_area_south_m2", 0),
        "k OstWest": dach_k[sc].get("gabled_area_eastwest_m2", 0),
        "k Flach":  dach_k[sc].get("flat_total", 0),
        "Dach-Kombis": dach_kombis_je_typ[sc],
    })
kombinations_zeilen.append({
    "Groessenklasse": "GESAMT",
    "Bedarf-Kombis": sum(bedarf_kombis_je_typ.values()),
    "Dach-Kombis": sum(dach_kombis_je_typ.values()),
})

excel_pfad = os.path.join(OUT_DIR, "Clustering_summary.xlsx")
with pd.ExcelWriter(excel_pfad, engine="openpyxl") as writer:
    ergebnis_uebersicht.to_excel(writer, sheet_name="Uebersicht", index=False)
    pd.DataFrame(kombinations_zeilen).to_excel(writer, sheet_name="Kombinationen", index=False)
    for sc in SIZE_CLASSES:
        sc_stats = alle_stats_df[alle_stats_df["size_class"] == sc]
        if not sc_stats.empty:
            sc_stats.to_excel(writer, sheet_name=f"Stats_{sc}", index=False)
    alle_stats_df.to_excel(writer, sheet_name="Alle_Stats", index=False)

print(f"gespeichert: {excel_pfad}")

gespeichert: ./nrw_results/Clustering_summary.xlsx
